# Creating Per League Per Season CSV 

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [3]:
HERE = Path.cwd()
REPO_ROOT = HERE.parents[3]
IN_CSV = HERE / "first_second_win_pct_all_seeds.csv"
OUT_CSV = HERE / "spearman_first_second_by_league_season.csv"

In [4]:
df = pd.read_csv(IN_CSV)

base_cols = ["league", "season", "team"]
seed_first = [c for c in df.columns if c.startswith("first_half_win_pct_seed_")]
seed_second = [c for c in df.columns if c.startswith("second_half_win_pct_seed_")]

df.head(2)

,league,season,team,first_half_win_pct_seed_1,second_half_win_pct_seed_1,first_half_win_pct_seed_2,second_half_win_pct_seed_2,first_half_win_pct_seed_3,second_half_win_pct_seed_3,first_half_win_pct_seed_4,...,first_half_win_pct_seed_7,second_half_win_pct_seed_7,first_half_win_pct_seed_8,second_half_win_pct_seed_8,first_half_win_pct_seed_9,second_half_win_pct_seed_9,first_half_win_pct_seed_10,second_half_win_pct_seed_10,first_half_win_pct_avg,second_half_win_pct_avg
0,bundesliga,2004,BAY,0.647059,0.235294,0.411765,0.235294,0.470588,0.235294,0.588235,...,0.411765,0.470588,0.470588,0.470588,0.294118,0.294118,0.529412,0.352941,0.476471,0.347059
1,bundesliga,2004,BIE,0.470588,0.470588,0.352941,0.470588,0.470588,0.411765,0.588235,...,0.470588,0.235294,0.529412,0.176471,0.647059,0.294118,0.117647,0.352941,0.447059,0.347059


In [5]:
def spearman_for_seed(group: pd.DataFrame, seed_num: int):
    """
    Compute Spearman r,p across teams within a (league, season) group
    for the given seed_num, using:
        first_half_win_pct_seed_{n}  vs  second_half_win_pct_seed_{n}
    Also return n_teams in this group after dropna.
    """
    xcol = f"first_half_win_pct_seed_{seed_num}"
    ycol = f"second_half_win_pct_seed_{seed_num}"

    # Keep team so we can count teams after dropna
    cols = ["team", xcol, ycol]

    cols = [c for c in cols if c in group.columns]
    sub = group[cols].dropna()

    # Count distinct teams with valid data for this seed
    n_teams = sub["team"].nunique() if "team" in sub.columns else len(sub)

    if n_teams < 2:
        return np.nan, np.nan, n_teams

    r, p = stats.spearmanr(sub[xcol].to_numpy(), sub[ycol].to_numpy())
    return float(r), float(p), int(n_teams)

In [6]:
records = []
for (league, season), g in df.groupby(["league", "season"], sort=True):
    rec = {"league": league, "season": season}

    # compute for each seed 
    n_teams_val = None
    for n in range(1, 11):
        r, p, n_teams = spearman_for_seed(g, n)
        rec[f"spearman_r_seed_{n}"] = r
        rec[f"spearman_p_seed_{n}"] = p
        n_teams_val = n_teams if n_teams_val is None else n_teams_val

    rec["n_teams"] = n_teams_val
    records.append(rec)

spearman_df = pd.DataFrame(records).sort_values(["league", "season"]).reset_index(drop=True)



Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [7]:
r_cols = [f"spearman_r_seed_{n}" for n in range(1, 11)]
p_cols = [f"spearman_p_seed_{n}" for n in range(1, 11)]

spearman_df["spearman_r_avg"] = spearman_df[r_cols].mean(axis=1, skipna=True)
spearman_df["spearman_p_avg"] = spearman_df[p_cols].mean(axis=1, skipna=True)

ordered_cols = ["league", "season", "n_teams"]
for n in range(1, 11):
    ordered_cols += [f"spearman_r_seed_{n}", f"spearman_p_seed_{n}"]
ordered_cols += ["spearman_r_avg", "spearman_p_avg"]

spearman_df = spearman_df[ordered_cols]

spearman_df.to_csv(OUT_CSV, index=False)
print(f"✅ Saved: {OUT_CSV}")
spearman_df.head()

✅ Saved: /Users/adhvik_rayaprolu/Desktop/uiuc/IML_Fall2025/skillvsluck/output/european_soccer_leagues/correlations/pure_luck_result_based/spearman_first_second_by_league_season.csv


,league,season,n_teams,spearman_r_seed_1,spearman_p_seed_1,spearman_r_seed_2,spearman_p_seed_2,spearman_r_seed_3,spearman_p_seed_3,spearman_r_seed_4,...,spearman_r_seed_7,spearman_p_seed_7,spearman_r_seed_8,spearman_p_seed_8,spearman_r_seed_9,spearman_p_seed_9,spearman_r_seed_10,spearman_p_seed_10,spearman_r_avg,spearman_p_avg
0,bundesliga,2004,18,0.212654,0.396886,-0.485857,0.040934,-0.263722,0.290320,0.120380,...,0.127808,0.613282,0.287110,0.248016,-0.016412,0.948466,0.279047,0.262134,0.047291,0.480749
1,bundesliga,2005,18,-0.166052,0.510206,0.033086,0.896306,0.022445,0.929558,0.209547,...,-0.150945,0.549925,-0.354062,0.149442,0.068060,0.788440,0.430108,0.074817,-0.003703,0.490231
2,bundesliga,2006,18,0.058041,0.819055,0.255535,0.306104,-0.119214,0.637527,0.302948,...,0.000000,1.000000,0.235023,0.347846,0.060022,0.812980,-0.487812,0.040010,0.007489,0.543122
3,bundesliga,2007,18,-0.114699,0.650411,0.361578,0.140390,-0.174152,0.489482,-0.028419,...,0.014216,0.955353,0.114231,0.651750,-0.362827,0.138925,0.296358,0.232426,-0.015139,0.539229
4,bundesliga,2008,18,-0.136502,0.589134,-0.235988,0.345812,0.025811,0.919025,-0.091054,...,-0.384251,0.115406,0.103449,0.682917,-0.292820,0.238314,-0.113381,0.654190,-0.154854,0.515082


# Creating Per League CSV

In [8]:
per_league_records = []
for lg, grp in spearman_df.groupby("league", sort=True):
    rec = {"league": lg}

    # mean across seasons for each seed
    for n in range(1, 11):
        rec[f"spearman_r_seed_{n}"] = grp[f"spearman_r_seed_{n}"].mean(skipna=True)
        rec[f"spearman_p_seed_{n}"] = grp[f"spearman_p_seed_{n}"].mean(skipna=True)

    # compute average of all 10 seeds' r and p per league
    r_cols = [f"spearman_r_seed_{i}" for i in range(1, 11)]
    p_cols = [f"spearman_p_seed_{i}" for i in range(1, 11)]
    rec["spearman_r_avg"] = np.nanmean([rec[c] for c in r_cols])
    rec["spearman_p_avg"] = np.nanmean([rec[c] for c in p_cols])

    per_league_records.append(rec)

per_league_df = pd.DataFrame(per_league_records).sort_values("league").reset_index(drop=True)

In [9]:
per_league_cols = ["league"]
for n in range(1, 11):
    per_league_cols += [f"spearman_r_seed_{n}", f"spearman_p_seed_{n}"]
per_league_cols += ["spearman_r_avg", "spearman_p_avg"]

per_league_df = per_league_df[per_league_cols]

OUT_CSV_LEAGUE = HERE / "spearman_first_second_by_league.csv"
per_league_df.to_csv(OUT_CSV_LEAGUE, index=False)
print(f"✅ Saved per-league file: {OUT_CSV_LEAGUE}")
per_league_df

✅ Saved per-league file: /Users/adhvik_rayaprolu/Desktop/uiuc/IML_Fall2025/skillvsluck/output/european_soccer_leagues/correlations/pure_luck_result_based/spearman_first_second_by_league.csv


,league,spearman_r_seed_1,spearman_p_seed_1,spearman_r_seed_2,spearman_p_seed_2,spearman_r_seed_3,spearman_p_seed_3,spearman_r_seed_4,spearman_p_seed_4,spearman_r_seed_5,...,spearman_r_seed_7,spearman_p_seed_7,spearman_r_seed_8,spearman_p_seed_8,spearman_r_seed_9,spearman_p_seed_9,spearman_r_seed_10,spearman_p_seed_10,spearman_r_avg,spearman_p_avg
0,bundesliga,-0.067387,0.509916,-0.027840,0.491348,-0.010276,0.585033,-0.020144,0.414654,-0.016263,...,-0.041760,0.477582,0.033563,0.540826,0.053749,0.483360,-0.037906,0.423910,-0.023836,0.493753
1,la_liga,0.004310,0.399505,-0.057592,0.587441,-0.012727,0.528172,-0.018572,0.600446,0.011029,...,-0.037747,0.539075,-0.023494,0.444086,0.088037,0.511409,0.045647,0.510807,0.009509,0.520702
2,premier_league,0.098803,0.497170,-0.044577,0.609582,-0.067730,0.558519,-0.001372,0.538688,-0.021113,...,0.040468,0.465772,0.008154,0.543746,-0.000206,0.509233,0.049491,0.625874,-0.000502,0.545751
3,serie_a,0.034981,0.460179,-0.068835,0.534015,-0.066340,0.586621,-0.104244,0.391253,-0.046037,...,-0.001013,0.475844,0.029676,0.485646,-0.041584,0.483387,-0.001980,0.630153,-0.026062,0.510635
